In [1]:
import tensorflow as tf

from tensorflow_model_optimization.python.core.keras.compat import keras
import numpy as np


I0000 00:00:1781134189.750271   78401 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781134195.159327   78401 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781134233.670900   78401 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# 1. Define the Keras Model Architecture
# PyTorch uses (channels, height, width), but Keras defaults to (height, width, channels)
model = keras.Sequential([
    # Input Layer (32x32 images with 3 channels)
    keras.layers.Input(shape=(32, 32, 3)),
    
    # conv1
    keras.layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same'),
    # conv2
    keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same'),
    # conv3
    keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same'),
    # conv4
    keras.layers.Conv2D(filters=48, kernel_size=(3, 3), activation='relu', padding='same'),
    # conv5
    keras.layers.Conv2D(filters=48, kernel_size=(3, 3), activation='relu', padding='same'),
    # conv6
    keras.layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same'),
    
    # conv7: FusedMaxPoolConv2dReLU (Pooling is applied BEFORE the convolution in ai8x)
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(filters=16, kernel_size=(3, 3), activation='relu', padding='same'),
    
    # conv8: FusedMaxPoolConv2dReLU
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(filters=8, kernel_size=(3, 3), activation='relu', padding='same'),
    
    # Flatten the 3D tensor to 1D before passing to Dense layer
    keras.layers.Flatten(),
    
    # Fully Connected Layer (fc) - 10 output classes for CIFAR-10
    keras.layers.Dense(10) 
])

# Display the model summary to verify the shape transitions
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 32, 32, 32)        896       
                                                                 
 conv2d_1 (Conv2D)           (None, 32, 32, 64)        18496     
                                                                 
 conv2d_2 (Conv2D)           (None, 32, 32, 64)        36928     
                                                                 
 conv2d_3 (Conv2D)           (None, 32, 32, 48)        27696     
                                                                 
 conv2d_4 (Conv2D)           (None, 32, 32, 48)        20784     
                                                                 
 conv2d_5 (Conv2D)           (None, 32, 32, 32)        13856     
                                                                 
 max_pooling2d (MaxPooling2  (None, 16, 16, 32)        0

I0000 00:00:1781134267.340389   78401 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20861 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:01:00.0, compute capability: 8.9


In [4]:
import datetime
from pathlib import Path
# 1. Load Data
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Shuffle and split Training Data (50,000 total) into Train (45,000) and Val (5,000)

rng = np.random.default_rng(seed=42)
indices = rng.permutation(len(x_train_full))

x_train_full = x_train_full[indices]
y_train_full = y_train_full[indices]

x_train = x_train_full[:45000]
y_train = y_train_full[:45000]
x_val = x_train_full[45000:]
y_val = y_train_full[45000:]

#remove unnecessary dimensions
y_train = y_train.squeeze()
y_val = y_val.squeeze()
y_test = y_test.squeeze()



# Normalize pixel values
x_train = x_train.astype("float32")
x_val = x_val.astype("float32")
x_test = x_test.astype("float32")

# Define Transformations
train_transform = keras.Sequential([
    keras.layers.ZeroPadding2D(padding=4),
    keras.layers.RandomCrop(height=32, width=32),
    keras.layers.RandomFlip("horizontal"),
    keras.layers.Rescaling(scale=1.0 / 128.0, offset=-1.0),

])
val_transform = keras.layers.Rescaling(scale=1.0 / 128.0, offset=-1.0)

BATCH_SIZE = 128

# 3. Create Datasets
train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(buffer_size=50000)
    .batch(BATCH_SIZE)
    .map(lambda x, y: (train_transform(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(BATCH_SIZE)
    .map(lambda x, y: (val_transform(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(BATCH_SIZE)
    .map(lambda x, y: (val_transform(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

LOG_DIR = Path("logs_small_model") / "fit" / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=str(LOG_DIR),
    histogram_freq=1,      # logs weight histograms every epoch
    write_graph=True,
    write_images=False
)

best_model_callback = keras.callbacks.ModelCheckpoint(
    filepath="cifar130k_best.keras",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

# def make_step_decay_schedule(decay_epochs, factor):
#     decay_epochs = set(decay_epochs)

#     def schedule(epoch, lr):
#         if epoch in decay_epochs:
#             return lr * factor
#         return lr

#     return schedule

# lr_callback = keras.callbacks.LearningRateScheduler(
#     make_step_decay_schedule(decay_epochs=[100, 150, 90], factor=0.1),
#     verbose=1
# )




opt = keras.optimizers.Adam(learning_rate=2e-4)
# 4. Compile and Train
model.compile(
    optimizer=opt,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.fit(train_dataset,
          epochs=150,
          validation_data=val_dataset,
          callbacks=[tensorboard_callback,
          best_model_callback])

model.save("cifar130k_last.keras")
# 5. Final Evaluation (Independent Test Set)
test_loss, test_acc = model.evaluate(test_dataset, verbose=2)
print(f"\nFinal Test Accuracy: {test_acc:.4f}")


Epoch 1/150


I0000 00:00:1781134348.492426   78918 cuda_dnn.cc:461] Loaded cuDNN version 92300
I0000 00:00:1781134351.423342   78921 service.cc:153] XLA service 0x75b6c4b982f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781134351.423380   78921 service.cc:161]   StreamExecutor [0]: NVIDIA L4, Compute Capability 8.9 (Driver: 12.7.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.0)
I0000 00:00:1781134351.591336   78921 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1781134351.905941   78921 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


352/352 [==============================] - ETA: 0s - loss: 1.9778 - accuracy: 0.2657
Epoch 1: val_accuracy improved from -inf to 0.37880, saving model to cifar130k_best.keras
352/352 [==============================] - 16s 17ms/step - loss: 1.9778 - accuracy: 0.2657 - val_loss: 1.7034 - val_accuracy: 0.3788
Epoch 2/150
349/352 [============================>.] - ETA: 0s - loss: 1.6869 - accuracy: 0.3745
Epoch 2: val_accuracy improved from 0.37880 to 0.42200, saving model to cifar130k_best.keras
352/352 [==============================] - 5s 15ms/step - loss: 1.6868 - accuracy: 0.3742 - val_loss: 1.5892 - val_accuracy: 0.4220
Epoch 3/150
349/352 [============================>.] - ETA: 0s - loss: 1.5739 - accuracy: 0.4206
Epoch 3: val_accuracy improved from 0.42200 to 0.47540, saving model to cifar130k_best.keras
352/352 [==============================] - 5s 15ms/step - loss: 1.5731 - accuracy: 0.4208 - val_loss: 1.4542 - val_accuracy: 0.4754
Epoch 4/150
349/352 [===========================

In [16]:
model_path = "cifar130k_best.keras"
model = keras.models.load_model(model_path)
test_loss, test_acc = model.evaluate(test_dataset, verbose=2)
print(f"\nFinal Test Accuracy: {test_acc:.4f}")

79/79 - 1s - loss: 0.5740 - accuracy: 0.8154 - 799ms/epoch - 10ms/step

Final Test Accuracy: 0.8154


In [17]:
# Regular post-training quantization (PTQ) from the trained float model.
# This does not run QAT; it only calibrates int8 ranges using representative data.

ptq_float_model = model

def ptq_representative_data_gen():
    for images, _ in val_dataset.unbatch().batch(1).take(100):
        yield [images]


ptq_converter = tf.lite.TFLiteConverter.from_keras_model(ptq_float_model)
ptq_converter.optimizations = [tf.lite.Optimize.DEFAULT]
ptq_converter.representative_dataset = ptq_representative_data_gen
ptq_converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
ptq_converter.inference_input_type = tf.int8
ptq_converter.inference_output_type = tf.int8

print("Converting float model with regular PTQ...")
ptq_tflite_model = ptq_converter.convert()
ptq_tflite_output_path = "cifar130k_ptq_int8.tflite"
with open(ptq_tflite_output_path, "wb") as f:
    f.write(ptq_tflite_model)

print(f"Success! PTQ int8 model saved to: {ptq_tflite_output_path}")

Converting float model with regular PTQ...
INFO:tensorflow:Assets written to: /tmp/tmp2i_5qhqe/assets


INFO:tensorflow:Assets written to: /tmp/tmp2i_5qhqe/assets
/workspace/embedded_development/tfdevrunpod/.venv/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1781138094.849381   78401 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1781138094.849400   78401 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1781138094.849638   78401 reader.cc:83] Reading SavedModel from: /tmp/tmp2i_5qhqe
I0000 00:00:1781138094.850481   78401 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1781138094.850486   78401 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp2i_5qhqe
I0000 00:00:1781138094.858051   78401 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1781138094.859222   78401 loader.cc:236] Restoring SavedModel bundle.
I0000 

Success! PTQ int8 model saved to: cifar130k_ptq_int8.tflite


fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1781138096.134039   78401 flatbuffer_export.cc:3851] Skipping runtime version metadata in the model. This will be generated by the exporter.


In [18]:
import datetime
from pathlib import Path
import tensorflow_model_optimization as tfmot

quantize_model = tfmot.quantization.keras.quantize_model



# q_aware stands for quantization aware.
q_aware_model = quantize_model(model)

# TensorBoard logs for QAT run
QAT_LOG_DIR = Path("logs_small_model") / "fit" / ("qat-" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

qat_tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=str(QAT_LOG_DIR),
    histogram_freq=1,
    write_graph=True,
    write_images=False
)

# `quantize_model` requires a recompile.
q_aware_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

q_aware_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer (QuantizeLa  (None, 32, 32, 3)         3         
 yer)                                                            
                                                                 
 quant_conv2d (QuantizeWrap  (None, 32, 32, 32)        963       
 perV2)                                                          
                                                                 
 quant_conv2d_1 (QuantizeWr  (None, 32, 32, 64)        18627     
 apperV2)                                                        
                                                                 
 quant_conv2d_2 (QuantizeWr  (None, 32, 32, 64)        37059     
 apperV2)                                                        
                                                                 
 quant_conv2d_3 (QuantizeWr  (None, 32, 32, 48)        2

In [19]:
best_model_callback = keras.callbacks.ModelCheckpoint(
    filepath="qat_cifar130k_best.keras",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)
q_aware_model.fit(
    train_dataset,
    epochs=100,
    validation_data=val_dataset,
    callbacks=[qat_tensorboard_callback,
    best_model_callback]
)
q_aware_model.save("./cifar130k_QAT.keras")


Epoch 1/100
351/352 [============================>.] - ETA: 0s - loss: 0.4212 - accuracy: 0.8529
Epoch 1: val_accuracy did not improve from 0.81940
352/352 [==============================] - 13s 27ms/step - loss: 0.4213 - accuracy: 0.8528 - val_loss: 0.5744 - val_accuracy: 0.8136
Epoch 2/100
352/352 [==============================] - ETA: 0s - loss: 0.4016 - accuracy: 0.8588
Epoch 2: val_accuracy did not improve from 0.81940
352/352 [==============================] - 9s 26ms/step - loss: 0.4016 - accuracy: 0.8588 - val_loss: 0.5879 - val_accuracy: 0.8126
Epoch 3/100
351/352 [============================>.] - ETA: 0s - loss: 0.3972 - accuracy: 0.8617
Epoch 3: val_accuracy did not improve from 0.81940
352/352 [==============================] - 9s 27ms/step - loss: 0.3971 - accuracy: 0.8617 - val_loss: 0.5791 - val_accuracy: 0.8100
Epoch 4/100
351/352 [============================>.] - ETA: 0s - loss: 0.3842 - accuracy: 0.8649
Epoch 4: val_accuracy did not improve from 0.81940
352/352 [==

In [20]:
print("Initializing TFLite Converter...")
converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = ptq_representative_data_gen

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

print("Quantizing model...")
quantized_tflite_model = converter.convert()
tflite_output_path = "qat_cifar130k.tflite"
with open(tflite_output_path, "wb") as f:
    f.write(quantized_tflite_model)

print(f"Success! Full int8 model saved to: {tflite_output_path}")

Initializing TFLite Converter...
Quantizing model...
INFO:tensorflow:Assets written to: /tmp/tmp5gcv1yd2/assets


INFO:tensorflow:Assets written to: /tmp/tmp5gcv1yd2/assets
/workspace/embedded_development/tfdevrunpod/.venv/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1781139121.981868   78401 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1781139121.981882   78401 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1781139121.981988   78401 reader.cc:83] Reading SavedModel from: /tmp/tmp5gcv1yd2
I0000 00:00:1781139121.984768   78401 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1781139121.984775   78401 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp5gcv1yd2
I0000 00:00:1781139122.006987   78401 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1781139122.092455   78401 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp5gcv1

Success! Full int8 model saved to: qat_cifar130k.tflite


fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1781139122.506464   78401 flatbuffer_export.cc:3851] Skipping runtime version metadata in the model. This will be generated by the exporter.


In [21]:
# Compare regular float, QAT, and PTQ performance on the test set.
import os
import numpy as np


def evaluate_tflite_classifier(model_path, dataset):
    if not os.path.exists(model_path):
        return None

    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    input_scale, input_zero_point = input_details["quantization"]
    output_scale, output_zero_point = output_details["quantization"]

    accuracy = keras.metrics.SparseCategoricalAccuracy()
    loss_metric = keras.metrics.Mean()

    for images, labels in dataset.unbatch().batch(1):
        input_data = images.numpy()
        if np.issubdtype(input_details["dtype"], np.integer):
            input_data = input_data / input_scale + input_zero_point
            input_data = np.clip(
                np.rint(input_data),
                np.iinfo(input_details["dtype"]).min,
                np.iinfo(input_details["dtype"]).max,
            ).astype(input_details["dtype"])
        else:
            input_data = input_data.astype(input_details["dtype"])

        interpreter.set_tensor(input_details["index"], input_data)
        interpreter.invoke()

        logits = interpreter.get_tensor(output_details["index"])
        if np.issubdtype(output_details["dtype"], np.integer):
            logits = (logits.astype(np.float32) - output_zero_point) * output_scale

        labels = tf.reshape(labels, [-1])
        batch_loss = keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
        loss_metric.update_state(batch_loss)
        accuracy.update_state(labels, logits)

    return float(loss_metric.result().numpy()), float(accuracy.result().numpy())


results = []

regular_loss, regular_acc = model.evaluate(test_dataset, verbose=0)
results.append(("Regular float Keras", regular_loss, regular_acc))

if "q_aware_model" in globals():
    qat_loss, qat_acc = q_aware_model.evaluate(test_dataset, verbose=0)
    results.append(("QAT fake-quant Keras", qat_loss, qat_acc))

qat_tflite_metrics = evaluate_tflite_classifier("qat_cifar130k.tflite", test_dataset)
if qat_tflite_metrics is not None:
    results.append(("QAT int8 TFLite", *qat_tflite_metrics))
else:
    print("Skipping QAT int8 TFLite: q8_model.tflite not found. Run the QAT conversion cell first.")

ptq_tflite_metrics = evaluate_tflite_classifier("cifar130k_ptq_int8.tflite", test_dataset)
if ptq_tflite_metrics is not None:
    results.append(("PTQ int8 TFLite", *ptq_tflite_metrics))
else:
    print("Skipping PTQ int8 TFLite: cifar400k_ptq_int8.tflite not found. Run the PTQ conversion cell first.")

print("\nTest-set comparison")
print(f"{'Model':<24} {'Loss':>10} {'Accuracy':>10}")
print("-" * 48)
for name, loss, acc in results:
    print(f"{name:<24} {loss:>10.4f} {acc:>10.4f}")

/workspace/embedded_development/tfdevrunpod/.venv/lib/python3.11/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.



Test-set comparison
Model                          Loss   Accuracy
------------------------------------------------
Regular float Keras          0.5740     0.8154
QAT fake-quant Keras         0.6242     0.8222
QAT int8 TFLite              0.6237     0.8234
PTQ int8 TFLite              0.5759     0.8156


In [4]:
import numpy as np

# Assuming you just ran: 
# (x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Save them to a single file (fast, uncompressed)
np.savez('cifar10_data.npz', 
         x_train_full=x_train_full, 
         y_train_full=y_train_full, 
         x_test=x_test, 
         y_test=y_test)

print("Saved instantly!")

Saved instantly!


In [ ]:
import tensorflow_model_optimization as tfmot

quantize_model = tfmot.quantization.keras.quantize_model

# q_aware stands for for quantization aware.
q_aware_model = quantize_model(model)

# `quantize_model` requires a recompile.
q_aware_model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

q_aware_model.summary()